In [8]:
!pip install requests tqdm faiss-cpu transformers tensorflow sentence-transformers textblob gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 27.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 43.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gensim]


In [9]:
import os
import requests
import zipfile
from tqdm import tqdm
from pathlib import Path

DATA_DIR = Path("./textbooks")

def download_and_extract_zip(url, extract_to=DATA_DIR):
  extract_to.mkdir(parents=True, exist_ok=True)

  zip_path = extract_to / "textbooks.zip"
  print("Downloading data...")

  response = requests.get(url, stream=True)

  with open(zip_path, "wb") as file:
    for chunk in tqdm(response.iter_content(chunk_size=1024), unit='KB'):
      if chunk:
        file.write(chunk)

  print("Extracting data...")
  with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_to)
  print("Data downloaded and extracted")

def load_text_files(directory):
  text_data = []
  file_paths = []

  for filepath in Path(directory).rglob("*.txt"):
    with open(filepath, "r", encoding="utf-8") as file:
      text = file.read()
      text_data.append(text)
      file_paths.append(filepath)

  return text_data, file_paths

def setup_environment():

  data_url = "https://www.dropbox.com/scl/fi/54p9kkx5n93bffyx08eba/textbooks.zip?rlkey=2y2c5x8y0uncnddichn9cmd7n&st=m290nmkk&dl=1"

  download_and_extract_zip(data_url)

  text_files_directory = DATA_DIR / "textbooks/en"
  texts, paths = load_text_files(text_files_directory)

  print("processing complete. Loaded {} documents.".format(len(paths)))
  return texts, paths


texts, document_paths = setup_environment()

88121KB [00:06, 13470.07KB/s]


Extracting data...
Data downloaded and extracted
processing complete. Loaded 18 documents.


In [15]:
import re
from gensim.utils import simple_preprocess
from textblob import TextBlob

def load_text_files(directory):
  texts = []
  for file_path in Path(directory).glob("F*.txt"):
    with open(file_path, "r", encoding="utf-8") as file:
      texts.append(file.read())
  return texts

def clean_and_tokenize(text):
  text = re.sub(r'\s+', ' ', text)
  text = text.lower()
  text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

  tokens = simple_preprocess(text)

  return ' '.join(tokens)


def correct_spelling(text):
  return text
  # return str(TextBlob(text).correct())

def chunk_text(text, chunk_size=200):
  words = text.split()
  return [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]

documents = load_text_files(DATA_DIR / "textbooks/en")

cleaned_documents = [clean_and_tokenize(doc) for doc in documents]

corrected_documents = [correct_spelling(doc) for doc in cleaned_documents]

chunked_documents = []

for doc in corrected_documents:
  chunked_documents.extend(chunk_text(doc))

print(f"Total document chunks created: {len(chunked_documents)}")


Total document chunks created: 1086


In [16]:
import torch
from transformers import AutoModel, AutoTokenizer

import numpy as np

# Verify that PyTorch detects the GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Available devices:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

def get_embeddings_in_batch(texts, batch_size=16):
  if not texts:
    return np.empty((0, model.config.hidden_size), dtype=np.float32)

  all_embeddings = []
  for i in range(0, len(texts), batch_size):
    batch_texts = texts[i:i+batch_size]

    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512,
    ).to(device)

    with torch.no_grad():
      print(i)
      outputs = model(**inputs)
      batch_embeddings = torch.mean(outputs.last_hidden_state, dim=1).cpu().numpy()

    all_embeddings.append(batch_embeddings)
  return np.vstack(all_embeddings)

embeddings = get_embeddings_in_batch(chunked_documents, batch_size=128)
print(f"Generated embeddings for {len(embeddings)} document chunks.")


Available devices: CPU only


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

0
128
256
384
512
640
768
896
1024
Generated embeddings for 1086 document chunks.


In [18]:
import faiss
import numpy as np

dimension = 384
index = faiss.IndexFlatL2(dimension)

embedding_matrix = np.array([embedding.flatten() for embedding in embeddings]).astype('float32')

index.add(embedding_matrix)

print(f"Total Embeddings indexed: {index.ntotal}")

Total Embeddings indexed: 1086


In [20]:
import faiss
import numpy as np
from transformers import AutoModel, AutoTokenizer
import torch

retrieval_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
retrieval_model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2").cpu()


def get_query_embedding(query):
  with torch.no_grad():
    inputs = retrieval_tokenizer(query, return_tensors="pt", padding=True, truncateion=True)
    outputs = retrieval_model(**inputs)
    embedding = torch.mean(outputs.last_hidden_state, dim=1).cpu().numpy()
  return embedding

embedding_dim = 384
index = faiss.IndexFlatL2(embedding_dim)

def retrieve_documents(query, top_k=5):
  query_embedding = get_query_embedding(query).astype("float32")
  distances, indices = index.search(query_embedding, top_k)
  results = [chunked_documents[idx] for idx in indices[0]]
  return results

sample_query = "Whar are the symptoms of high blood pressure?"

similar_documents = retrieve_documents(sample_query)

print("retrieved documents:", similar_documents)







Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

retrieved documents: ['crisis vestibuloocular ref ex scan aureus ventricular septal defect ventricular tachycardia von willebrands disease von willebrand factor varicellazoster virus white blood cell world health organization hemoglobin plasma mean corpuscular hemoglobin platelet count prothrombin time reticulocyte count sedimentation rate erythrocyte westergren proteins total mgdl moll mm pgcell fmolcell mm seconds seconds of red cells male mmh mmh female mmh mmh mg', 'crisis vestibuloocular ref ex scan aureus ventricular septal defect ventricular tachycardia von willebrands disease von willebrand factor varicellazoster virus white blood cell world health organization hemoglobin plasma mean corpuscular hemoglobin platelet count prothrombin time reticulocyte count sedimentation rate erythrocyte westergren proteins total mgdl moll mm pgcell fmolcell mm seconds seconds of red cells male mmh mmh female mmh mmh mg', 'crisis vestibuloocular ref ex scan aureus ventricular septal defect ventr

In [22]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

generation_tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct", trust_remote_code=True)
generation_model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3.5-mini-instruct", trust_remote_code=True)
generation_model.to(device)



Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
          (rotary_emb): Phi3LongRoPEScaledRotaryEmbedding()
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm()
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
        (post_attention_layernorm): Phi3RMSNorm()
      )
    )
    (norm): Phi3RMSNorm()
  )
  (lm_head): Linear(in_features

In [28]:
def generate_response(query, context, max_new_tokens=100):
  input_text = f"User query: {query} \n\nContext:\n{context}\n\nAnswer:"

  inputs = generation_tokenizer(input_text, return_tensors="pt", padding=True, truncation=True).to(device)
  print(inputs["input_ids"])
  with torch.no_grad():
    outputs = generation_model.generate(
        inputs["input_ids"],
        max_new_tokens=max_new_tokens,
        num_return_sequences=1,
        use_cache=False
    )
  print("12121")
  response_text = generation_tokenizer.decode(outputs[0], skip_special_tokens=True)
  return response_text

retrieved_text = " ".join(similar_documents)
response = generate_response(sample_query, retrieved_text)

print("Generated response:", response)

tensor([[ 4911,  2346, 29901,   806,   279,   526,   278, 25828,  4835,   310,
          1880, 10416, 12959, 29973, 29871,    13,    13,  2677, 29901,    13,
         29883,  3780,   275, 16779,   747,  7207,   542,  1070,  2143,   429,
         12812,   263,   545,   375,  9712,  2200,  1070,  4843,   284, 23503,
          9712,  2200,  1070,   260,   496, 29891,  7543,   423,  1005,   281,
          1924,  1182,  4167, 17135,  1005,   281,  1924, 16472,  7329,   722,
           293,  3547, 29920, 15664, 24424,  4796, 10416,  3038,  3186,  9045,
         13013,  9736,   468,   417,  2109,   715, 25392,  2099,  1034, 13364,
         16637,  9736,   468,   417,  2109, 15284,  1026,  2302,   410,   386,
           456,  2109,   931,  3240, 12906, 22502,   371,  2302,  7048,  2073,
           362,  6554,   604,  1541,   307,  1270,   371,   281,  4156, 20378,
          3279,  1144,  3001,   286, 29887, 11671,   286,  3028,  5654, 23822,
          3729,   285, 29885,   324,  3729,  5654,  

/usr/local/lib/python3.13/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.13/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


KeyboardInterrupt: 